# Experiment 1 — Univariate LSTM Baseline

Goal: establish a performance floor using only PM2.5 history (autocorrelation).

In [ ]:
# Core libraries
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
plt.style.use("seaborn-v0_8")

In [ ]:
# Performance: GPU + mixed precision + XLA
tf.keras.mixed_precision.set_global_policy("mixed_float16")
tf.config.optimizer.set_jit(True)
print("Mixed precision:", tf.keras.mixed_precision.global_policy())

In [ ]:
# Paths
BASE_DIR = Path('.')
TRAIN_PATH = BASE_DIR / 'train.csv'
TEST_PATH = BASE_DIR / 'test.csv'

train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

print('Train shape:', train_raw.shape)
print('Test shape:', test_raw.shape)

## Preprocessing + Feature Engineering
- Linear interpolation with 6-hour limit
- One-hot encode cbwd if categorical
- Cyclical time encodings

In [ ]:
def add_time_features(df):
    df = df.copy()
    df['hour'] = df['datetime'].dt.hour
    df['month'] = df['datetime'].dt.month
    df['dayofweek'] = df['datetime'].dt.dayofweek
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['dow_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)
    return df.drop(columns=['hour', 'month', 'dayofweek'])

def preprocess(df, is_train=True):
    df = df.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    df = df.sort_values('datetime')
    if 'cbwd' in df.columns:
        df['cbwd'] = df['cbwd'].fillna(method='ffill')
        df = pd.get_dummies(df, columns=['cbwd'], prefix='cbwd', drop_first=False)
    df = add_time_features(df)
    if 'No' in df.columns:
        df = df.drop(columns=['No'])
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].interpolate(limit=6, limit_direction='both')
    if is_train:
        df = df.dropna()
    else:
        df[num_cols] = df[num_cols].fillna(method='ffill').fillna(method='bfill')
    return df.reset_index(drop=True)

In [ ]:
train_df = preprocess(train_raw, is_train=True)
test_df = preprocess(test_raw, is_train=False)
print('Train after preprocessing:', train_df.shape)
print('Test after preprocessing:', test_df.shape)

In [ ]:
# Time-based split (80/20)
split_idx = int(len(train_df) * 0.8)
train_split = train_df.iloc[:split_idx].reset_index(drop=True)
val_split = train_df.iloc[split_idx:].reset_index(drop=True)

TARGET = 'pm2.5'
LOOKBACK = 48

def scale_train_val(train_df, val_df, feature_cols):
    scaler = MinMaxScaler()
    train_scaled = train_df.copy()
    val_scaled = val_df.copy()
    train_scaled[feature_cols] = scaler.fit_transform(train_df[feature_cols])
    val_scaled[feature_cols] = scaler.transform(val_df[feature_cols])
    return train_scaled, val_scaled, scaler

def make_sequences(df, feature_cols, target_col, lookback):
    data = df[feature_cols].values
    target = df[target_col].values
    X, y = [], []
    for i in range(lookback, len(df)):
        X.append(data[i - lookback : i])
        y.append(target[i])
    return np.array(X), np.array(y)

def evaluate_regression(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return rmse, mae, r2

def plot_predictions(y_true, y_pred, title):
    plt.figure(figsize=(10, 4))
    plt.plot(y_true, label='Actual', linewidth=1)
    plt.plot(y_pred, label='Predicted', linewidth=1)
    plt.title(title)
    plt.xlabel('Time')
    plt.ylabel('PM2.5')
    plt.legend()
    plt.tight_layout()
    plt.show()

## Train Univariate LSTM
- Input shape: $(48, 1)$
- Model: LSTM(50) → Dense(1)

In [ ]:
uni_features = [TARGET]
train_uni, val_uni, uni_scaler = scale_train_val(train_split, val_split, uni_features)

X_train_uni, y_train_uni = make_sequences(train_uni, uni_features, TARGET, LOOKBACK)
X_val_uni, y_val_uni = make_sequences(val_uni, uni_features, TARGET, LOOKBACK)

model_uni = models.Sequential([
    layers.LSTM(50, input_shape=(X_train_uni.shape[1], X_train_uni.shape[2])),
    layers.Dense(1)
])
model_uni.compile(optimizer='adam', loss='mse')

es = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history_uni = model_uni.fit(
    X_train_uni, y_train_uni,
    validation_data=(X_val_uni, y_val_uni),
    epochs=50, batch_size=64, callbacks=[es], verbose=1
)

y_val_pred = model_uni.predict(X_val_uni, verbose=0).ravel()
y_val_true = y_val_uni.ravel()

y_val_true_inv = uni_scaler.inverse_transform(y_val_true.reshape(-1, 1)).ravel()
y_val_pred_inv = uni_scaler.inverse_transform(y_val_pred.reshape(-1, 1)).ravel()

rmse, mae, r2 = evaluate_regression(y_val_true_inv, y_val_pred_inv)
print(f'Exp1 RMSE: {rmse:.3f} | MAE: {mae:.3f} | R2: {r2:.3f}')
plot_predictions(y_val_true_inv[:500], y_val_pred_inv[:500], 'Experiment 1 — Univariate LSTM')

In [ ]:
# Submission (Experiment 1)
def recursive_forecast(model, train_scaled, test_scaled, feature_cols, target_col, lookback):
    window = train_scaled[feature_cols].iloc[-lookback:].values
    preds_scaled = []
    for i in range(len(test_scaled)):
        x = window[np.newaxis, ...]
        y_pred = model.predict(x, verbose=0).ravel()[0]
        preds_scaled.append(y_pred)
        next_row = test_scaled[feature_cols].iloc[i].copy()
        next_row[target_col] = y_pred
        window = np.vstack([window[1:], next_row.values])
    return np.array(preds_scaled)

def save_submission(preds, sample_path, test_raw_df, out_path):
    submission = pd.read_csv(sample_path)
    if "row ID" in submission.columns:
        submission["pm2.5"] = preds[: len(submission)]
    else:
        submission = pd.DataFrame({"row ID": test_raw_df["datetime"].astype(str), "pm2.5": preds})
    submission.to_csv(out_path, index=False)
    print("Saved submission to:", out_path)

SAMPLE_SUB_PATH = BASE_DIR / "sample_submission .csv"

test_uni = test_df[[TARGET]].copy()
test_uni[TARGET] = train_split[TARGET].iloc[-1]
test_uni_scaled = test_uni.copy()
test_uni_scaled[TARGET] = uni_scaler.transform(test_uni[[TARGET]])

preds_scaled = recursive_forecast(
    model_uni, train_uni, test_uni_scaled, [TARGET], TARGET, LOOKBACK
 )
preds = uni_scaler.inverse_transform(preds_scaled.reshape(-1, 1)).ravel()
save_submission(preds, SAMPLE_SUB_PATH, test_raw, BASE_DIR / "submission_exp1.csv")